# O núcleo do código

### Quais arquivos de uma biblioteca eu não posso mexer sem quebrar tudo?

Você acabou de entrar num projeto com centenas de módulos. Ou vai adotar uma biblioteca e precisa saber no que está se metendo. A pergunta é sempre a mesma:

> **Onde eu posso mexer com segurança, e onde eu preciso ter medo?**

Este notebook responde isso com **um algoritmo linear** — a decomposição em k-core — rodando sobre o grafo de dependências de pacotes Python reais.

**O caminho:**

| | |
|---|---|
| **1** | transformar código em grafo (com 5 arquivos, para ver o mecanismo) |
| **2** | rodar a decomposição no `networkx` e ler o resultado |
| **3** | mostrar por que a resposta óbvia — "conte quem me importa" — erra |
| **4** | comparar uma biblioteca bem arquitetada com um monólito |
| **5** | transformar o número numa recomendação de manutenção |

**Pré-requisito:** saber o que é k-core, k-shell e degenerescência.

## 0 · Preparação

In [1]:
import importlib, importlib.util, subprocess, sys

for pacote, mod in [("networkx", "networkx"), ("plotly", "plotly"), ("pandas", "pandas")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

import ast, os, math, tempfile, textwrap
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# mantém o notebook leve: o JS do plotly vem da CDN em vez de embutido
pio.renderers.default = "plotly_mimetype+notebook_connected"

print("networkx", nx.__version__, "| pandas", pd.__version__)

networkx 3.0 | pandas 2.0.1


C:\Users\Miguel\AppData\Roaming\Python\Python311\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




In [ ]:
# ---------- paleta e funções de desenho ----------
AZUIS = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#1c5cab", "#104281", "#0d366b"]
LARANJA, CINZA_NO, CINZA_ARESTA, CINZA_TEXTO = "#eb6834", "#c3c2b7", "#d9d8d2", "#52514e"
SUPERFICIE = "#fcfcfb"

def rampa_azul(valores, vmin=None, vmax=None):
    v = np.asarray(list(valores), dtype=float)
    vmin = v.min() if vmin is None else vmin
    vmax = v.max() if vmax is None else vmax
    if vmax == vmin:
        return [AZUIS[-1]] * len(v)
    i = np.clip(((v - vmin) / (vmax - vmin) * (len(AZUIS) - 1)).round().astype(int), 0, len(AZUIS) - 1)
    return [AZUIS[j] for j in i]

def estilo(fig, titulo=None, altura=460):
    fig.update_layout(title=titulo, height=altura, paper_bgcolor=SUPERFICIE, plot_bgcolor=SUPERFICIE,
                      font=dict(family="system-ui, sans-serif", color=CINZA_TEXTO, size=13),
                      margin=dict(l=60, r=30, t=70 if titulo else 30, b=60))
    fig.update_xaxes(gridcolor="#e1e0d9", zeroline=False)
    fig.update_yaxes(gridcolor="#e1e0d9", zeroline=False)
    return fig

def traco_arestas(g, pos, largura=1.0, opacidade=0.7):
    xs, ys = [], []
    for u, v in g.edges():
        xs += [pos[u][0], pos[v][0], None]
        ys += [pos[u][1], pos[v][1], None]
    return go.Scatter(x=xs, y=ys, mode="lines", hoverinfo="skip", opacity=opacidade,
                      line=dict(color=CINZA_ARESTA, width=largura), showlegend=False)

def traco_nos(g, pos, cores, tam=14, hover=None, rotulos=True, textos=None):
    nos = list(g.nodes())
    lista = cores if isinstance(cores, list) else [cores] * len(nos)
    return go.Scatter(
        x=[pos[n][0] for n in nos], y=[pos[n][1] for n in nos],
        mode="markers+text" if rotulos else "markers",
        text=(textos if textos is not None else [str(n) for n in nos]) if rotulos else None,
        textposition="middle center",
        textfont=dict(color=[("#0b0b0b" if c in AZUIS[:3] or c == CINZA_NO else "white") for c in lista],
                      size=11),
        hovertext=hover if hover is not None else [str(n) for n in nos], hoverinfo="text",
        marker=dict(color=cores, size=tam, line=dict(color="#ffffff", width=1.5)), showlegend=False)

def figura_rede(tracos, titulo="", altura=520):
    fig = go.Figure(data=tracos)
    fig.update_layout(title=dict(text=titulo, x=0.02, font=dict(size=17, color="#0b0b0b")),
                      height=altura, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
                      font=dict(family="system-ui, sans-serif", color=CINZA_TEXTO, size=13),
                      margin=dict(l=10, r=10, t=50, b=10), hovermode="closest",
                      xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x", scaleratio=1))
    return fig

## 1 · De código a grafo

Um projeto de software **é** um grafo: cada módulo é um vértice, cada `import` é uma aresta. Três decisões de modelagem, e cada uma muda a resposta.

**1. Ler, não executar.** Usamos o módulo `ast` da biblioteca padrão para extrair os `import` de cada arquivo. Nada do código analisado roda — questão de segurança e de sanidade, já que importar um pacote inteiro tem efeitos colaterais.

**2. Só arestas internas.** `import numpy` dentro do NetworkX não vira aresta. Estamos medindo a arquitetura *interna* do pacote, não o ecossistema.

**3. Grafo não direcionado.** `import` tem direção, e nós a jogamos fora de propósito. O motivo: a pergunta é **"quem está preso com quem"**, e prisão é mútua. Se A importa B e B importa A, isso é *uma* ligação de dependência mútua — e é exatamente esse tipo de amarra que o k-core conta.

In [ ]:
def modulos_do_pacote(raiz, incluir_testes=False):
    """Mapeia caminho de arquivo -> nome de módulo pontilhado."""
    pacote = os.path.basename(os.path.normpath(raiz))
    achados = {}
    for pasta, subdirs, arquivos in os.walk(raiz):
        subdirs[:] = [x for x in subdirs if x not in {"__pycache__", ".git"}
                      and (incluir_testes or x not in {"tests", "test", "testing"})]
        for arq in arquivos:
            if not arq.endswith(".py"):
                continue
            rel = os.path.relpath(os.path.join(pasta, arq), raiz).replace(os.sep, ".")[:-3]
            if rel.endswith(".__init__"):
                rel = rel[:-9]
            elif rel == "__init__":
                rel = ""
            achados[os.path.join(pasta, arq)] = pacote + ("." + rel if rel else "")
    return pacote, achados


def imports_do_arquivo(caminho, nome_modulo, pacote):
    """Nomes importados por este arquivo, resolvidos para absolutos e filtrados ao pacote."""
    try:
        with open(caminho, encoding="utf-8") as fh:
            arvore = ast.parse(fh.read(), filename=caminho)
    except (SyntaxError, UnicodeDecodeError):
        return set()

    partes, alvos = nome_modulo.split("."), set()
    for no in ast.walk(arvore):
        if isinstance(no, ast.Import):
            alvos.update(a.name for a in no.names)
        elif isinstance(no, ast.ImportFrom):
            if no.level:                                   # from . / .. import x
                base = partes[:-no.level] if no.level <= len(partes) else []
                prefixo = ".".join(base + ([no.module] if no.module else []))
            else:
                prefixo = no.module or ""
            if prefixo:
                alvos.add(prefixo)
                alvos.update(prefixo + "." + a.name for a in no.names)
    return {a for a in alvos if a == pacote or a.startswith(pacote + ".")}


def grafo_de_dependencias(raiz, incluir_testes=False):
    """Grafo NÃO direcionado de dependências internas de um pacote Python."""
    pacote, mapa = modulos_do_pacote(raiz, incluir_testes)
    conhecidos = set(mapa.values())
    G = nx.Graph()
    G.add_nodes_from(conhecidos)
    for caminho, nome in mapa.items():
        for alvo in imports_do_arquivo(caminho, nome, pacote):
            if alvo in conhecidos and alvo != nome:
                G.add_edge(nome, alvo)
    return pacote, G


def raiz_do_pacote(nome):
    spec = importlib.util.find_spec(nome)
    if spec is None or not spec.submodule_search_locations:
        return None
    return list(spec.submodule_search_locations)[0]

### 1.1 · O mecanismo, em cinco arquivos

Antes de soltar isso em centenas de módulos, vale ver o mecanismo inteiro num pacote que cabe numa tela. Vamos **escrever** um pacote de cinco arquivos e rodar nele **exatamente as mesmas funções** de cima.

In [4]:
FONTES = {
    "__init__.py": "# Pacote de exemplo.\n"
                   "from demo.core import resolver\n"
                   "from demo.cli import main\n",
    "cli.py":      "import argparse\n"
                   "from .core import resolver\n"
                   "from .io_utils import ler\n",
    "core.py":     "from .model import Grafo\n"
                   "from .io_utils import ler, escrever\n",
    "model.py":    "from .core import normalizar\n"
                   "from .io_utils import escrever\n",
    "io_utils.py": "import os\n"
                   "import json\n",
}

RAIZ_DEMO = os.path.join(tempfile.mkdtemp(), "demo")
os.makedirs(RAIZ_DEMO, exist_ok=True)
for _nome, _src in FONTES.items():
    with open(os.path.join(RAIZ_DEMO, _nome), "w", encoding="utf-8") as fh:
        fh.write(_src)
    print(f"--- demo/{_nome} " + "-" * (44 - len(_nome)))
    print(textwrap.indent(_src.rstrip(), "    "))
print("-" * 60)

--- demo/__init__.py ---------------------------------
    # Pacote de exemplo.
    from demo.core import resolver
    from demo.cli import main
--- demo/cli.py --------------------------------------
    import argparse
    from .core import resolver
    from .io_utils import ler
--- demo/core.py -------------------------------------
    from .model import Grafo
    from .io_utils import ler, escrever
--- demo/model.py ------------------------------------
    from .core import normalizar
    from .io_utils import escrever
--- demo/io_utils.py ---------------------------------
    import os
    import json
------------------------------------------------------------


In [5]:
pacote_d, mapa_d = modulos_do_pacote(RAIZ_DEMO)
conhecidos_d = set(mapa_d.values())

print(f"{'módulo':<16}{'importa, do próprio pacote':<32}")
print("-" * 48)
arestas_dirigidas = 0
for caminho, nome in sorted(mapa_d.items(), key=lambda kv: kv[1]):
    alvos = sorted(a for a in imports_do_arquivo(caminho, nome, pacote_d)
                   if a in conhecidos_d and a != nome)
    arestas_dirigidas += len(alvos)
    print(f"{nome:<16}{', '.join(a.split('.')[-1] for a in alvos) or '— nada interno —':<32}")

_, Ud = grafo_de_dependencias(RAIZ_DEMO)
print(f"\nsetas de import (com direção) : {arestas_dirigidas}")
print(f"arestas do grafo (sem direção): {Ud.number_of_edges()}")
print(f"\nUma sumiu: core e model se importam MUTUAMENTE. Duas setas, uma aresta —")
print(f"e é justamente essa amarra mútua que o k-core conta.")

módulo          importa, do próprio pacote      
------------------------------------------------
demo            cli, core                       
demo.cli        core, io_utils                  
demo.core       io_utils, model                 
demo.io_utils   — nada interno —                
demo.model      core, io_utils                  

setas de import (com direção) : 8
arestas do grafo (sem direção): 7

Uma sumiu: core e model se importam MUTUAMENTE. Duas setas, uma aresta —
e é justamente essa amarra mútua que o k-core conta.


In [6]:
curto_d = lambda v: v.split(".")[-1] if "." in v else v
POS_D = {"demo": (0.0, 1.0), "demo.cli": (-1.0, 0.15), "demo.core": (0.0, 0.0),
         "demo.model": (1.0, 0.15), "demo.io_utils": (0.0, -1.0)}

quantos_importam = {n: sum(1 for c, m in mapa_d.items()
                           if n in imports_do_arquivo(c, m, pacote_d) and m != n)
                    for n in Ud.nodes()}
hover_d = [f"<b>{curto_d(n)}</b><br>importado por {quantos_importam[n]}<br>"
           f"grau no grafo: {Ud.degree(n)}" for n in Ud.nodes()]

fig = figura_rede([traco_arestas(Ud, POS_D, largura=2.0, opacidade=0.85),
                   traco_nos(Ud, POS_D, AZUIS[4], 48, hover_d,
                             textos=[curto_d(n) for n in Ud.nodes()])],
                  "demo · 5 módulos, 7 arestas", altura=420)
fig.show()

**O `io_utils` é o mais importado e não importa ninguém.** Três módulos dependem dele; ele não depende de nenhum. É uma **folha popular**: você pode reescrever o corpo dele à vontade, desde que a assinatura das funções continue igual. Ninguém volta pra te morder.

**O `core` e o `model` se seguram.** Formam o único par mútuo do pacote. Mexer num obriga a olhar o outro.

Repare que essas duas situações são **opostas em risco** e **indistinguíveis pelo grau**: `io_utils` tem grau 3, `core` tem grau 4. O número de vizinhos não separa "seguro" de "preso".

E uma observação: rodando `nx.core_number` neste grafo, sai **2 para todo mundo**. Cinco arquivos são poucos demais para ter camadas. A decomposição só passa a dizer algo interessante quando há hierarquia suficiente e isso normalmente pede escala.

## 2 · A decomposição num pacote real

O pacote analisado é o próprio **NetworkX**: está instalado, tem tamanho respeitável, e é código que você já usou sem olhar por dentro.

In [7]:
ALVO = "networkx"
raiz = raiz_do_pacote(ALVO)
pacote, G = grafo_de_dependencias(raiz)

curto = lambda v: v[len(pacote) + 1:] if v.startswith(pacote + ".") else v
core = nx.core_number(G)
grau = dict(G.degree())
d = max(core.values())
nucleo = sorted((v for v in G if core[v] == d), key=curto)

print(f"pacote            : {pacote}")
print(f"módulos (n)       : {G.number_of_nodes()}")
print(f"dependências (m)  : {G.number_of_edges()}")
print(f"DEGENERESCÊNCIA   : {d}")
print(f"núcleo principal  : {len(nucleo)} módulos "
      f"({100 * len(nucleo) / G.number_of_nodes():.1f}% do pacote)\n")
print("os módulos do núcleo:")
for v in nucleo:
    interno = sum(1 for u in G[v] if core[u] == d)
    print(f"   {curto(v):<26} grau total {grau[v]:>4}   vizinhos dentro do núcleo {interno}")

pacote            : networkx
módulos (n)       : 274
dependências (m)  : 681
DEGENERESCÊNCIA   : 5
núcleo principal  : 7 módulos (2.6% do pacote)

os módulos do núcleo:
   classes.digraph            grau total    7   vizinhos dentro do núcleo 5
   classes.graph              grau total    7   vizinhos dentro do núcleo 5
   classes.multidigraph       grau total    7   vizinhos dentro do núcleo 5
   classes.multigraph         grau total    6   vizinhos dentro do núcleo 5
   classes.reportviews        grau total    6   vizinhos dentro do núcleo 5
   convert                    grau total    5   vizinhos dentro do núcleo 5
   networkx                   grau total  227   vizinhos dentro do núcleo 6


**O que esses números dizem.**

A degenerescência é **5**: existe um grupo de módulos em que **cada um depende de pelo menos 5 outros do mesmo grupo**, e não existe grupo assim para 6. Esse grupo é o 5-core, e ele tem **7 módulos de 274 — 2,6% do pacote**.

E eles fazem sentido juntos: as quatro classes de grafo (`Graph`, `DiGraph`, `MultiGraph`, `MultiDiGraph`), o módulo de *views* que todas expõem, o `convert`, e o `__init__` do pacote. O `Graph.__init__` chama o `convert` para construir grafo a partir de dados; o `convert` constrói objetos `Graph`. É um ciclo real.

**Traduzindo para a pergunta original:** destes 274 arquivos, **7 são o território perigoso**. Mexer em qualquer um deles obriga a entender os outros 6.

In [8]:
tam_shell = pd.Series(core).value_counts().sort_index()
tam_core = pd.Series({k: sum(v >= k for v in core.values()) for k in range(0, d + 1)})

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("k-shell: quantos módulos PARARAM em cada k",
                                    "k-core: quantos módulos SOBREVIVEM até cada k"))
fig.add_bar(x=tam_shell.index, y=tam_shell.values, marker_color=rampa_azul(tam_shell.index, 0, d),
            text=tam_shell.values, textposition="outside",
            hovertemplate="%{x}-shell<br>%{y} módulos<extra></extra>", showlegend=False, row=1, col=1)
fig.add_bar(x=tam_core.index, y=tam_core.values, marker_color=rampa_azul(tam_core.index, 0, d),
            text=tam_core.values, textposition="outside",
            hovertemplate="%{x}-core<br>%{y} módulos<extra></extra>", showlegend=False, row=1, col=2)
fig.update_xaxes(title_text="k", dtick=1, row=1, col=1)
fig.update_xaxes(title_text="k", dtick=1, row=1, col=2)
fig.update_yaxes(title_text="módulos", row=1, col=1)
estilo(fig, f"{pacote}: a decomposição inteira", 430).show()

print("conferência — a soma das k-shells tem que dar o total de módulos:")
print(f"   soma das shells = {tam_shell.sum()}   |   módulos = {G.number_of_nodes()}")

conferência — a soma das k-shells tem que dar o total de módulos:
   soma das shells = 274   |   módulos = 274


### Como ler este gráfico de barras

São **duas leituras da mesma decomposição**, e a diferença entre elas é a diferença entre k-shell e k-core. Vale entender as duas, porque cada uma responde uma pergunta diferente.

**Painel da esquerda — k-shell: uma partição.**

Cada barra conta os módulos cuja coreness é **exatamente** aquele k, ou seja, os que **pararam** naquela camada: sobreviveram à descascada até k e caíram em k+1. Como cada módulo tem uma única coreness, **cada módulo aparece em exatamente uma barra** — e por isso a soma das barras dá 274, o pacote inteiro.

Leitura da forma: o pico está em **k=3, com 101 módulos**. À esquerda dele, a periferia; à direita, a queda até o núcleo de 7. É um **sino** com massa concentrada no meio-baixo, cauda fina à direita.

**Painel da direita — k-core: um encaixe cumulativo.**

Cada barra conta os módulos com coreness **maior ou igual** àquele k, ou seja, quem ainda **estava de pé** quando a descascada chegou em k. Aqui um mesmo módulo aparece em várias barras: quem tem coreness 5 é contado em todas, de 0 a 5. Por isso as barras só descem. Isso remete à propriedade de aninhamento, `(k+1)-core ⊆ k-core`, desenhada.

**A relação entre os dois painéis** é direta: cada barra da direita é a **soma de todas as barras da esquerda daquele k para cima**. O 4-core (61) é a 4-shell (54) mais a 5-shell (7). Entender isso, é também entender a decomposição.

**E por que isso importa para a pergunta que abriu o notebook?**

Porque o painel da direita é literalmente um **orçamento de risco**. Ele responde: *"se eu decidir que 'perigoso' significa coreness ≥ k, quantos arquivos eu tenho que tratar com cuidado?"*

| se eu chamar de perigoso… | tenho que cuidar de | do pacote |
|---|---|---|
| coreness ≥ 5 | 7 módulos | 2,6% |
| coreness ≥ 4 | 61 módulos | 22% |
| coreness ≥ 3 | 162 módulos | 59% |

O salto de 5 para 4 é o que interessa: **afrouxar o critério em um degrau multiplica o escopo por quase nove**. Isso diz que o 5-core é uma fronteira nítida, há um caroço pequeno e muito coeso, e depois uma queda. Se as barras descessem suavemente, não haveria fronteira natural e a escolha de k seria uma decisão sua, não do grafo.

### O grafo inteiro

Antes de saber a resposta: aqui está o pacote todo, colorido por **grau**. Quanto mais escuro, mais vizinhos. Tente achar os sete do núcleo.

In [9]:
POS = nx.spring_layout(G, seed=42, k=0.15, iterations=80)
nos = list(G.nodes())
hover_G = [f"<b>{curto(n)}</b><br>grau = {grau[n]}<br>coreness = {core[n]}" for n in nos]

figura_rede([traco_arestas(G, POS, largura=0.5, opacidade=0.35),
             traco_nos(G, POS, rampa_azul([grau[n] for n in nos]), 8, hover_G, rotulos=False)],
            f"{pacote} · {G.number_of_nodes()} módulos, {G.number_of_edges()} dependências — cor = grau",
            altura=600).show()

print("os 6 módulos de maior grau:")
for v in sorted(G, key=lambda v: -grau[v])[:6]:
    print(f"   {curto(v):<22} grau {grau[v]:>4}   coreness {core[v]}"
          f"{'   <-- núcleo' if core[v] == d else ''}")

os 6 módulos de maior grau:
   networkx               grau  227   coreness 5   <-- núcleo
   utils                  grau  129   coreness 4
   algorithms             grau   77   coreness 4
   generators             grau   28   coreness 4
   exception              grau   26   coreness 4
   utils.decorators       grau   16   coreness 4


Não dá para achar. E a lista abaixo da figura explica: dos seis módulos mais escuros, **um** está no núcleo. O `utils`, o segundo mais conectado do pacote inteiro, fica de fora.

Agora as mesmas posições e as mesmas arestas, mudando só a cor, que passa a ser a **coreness**. O menu no canto troca qual k-core fica destacado, e o contorno tracejado marca a região dele.

In [10]:
from scipy.spatial import ConvexHull

def casco(nos_sel, pos):
    pts = np.array([pos[n] for n in nos_sel])
    if len(pts) < 3:
        return None
    h = ConvexHull(pts)
    ciclo = list(h.vertices) + [h.vertices[0]]
    return pts[ciclo, 0], pts[ciclo, 1]

cores_para_k = lambda k: [AZUIS[6] if core[n] >= k else CINZA_NO for n in nos]

fig = figura_rede([traco_arestas(G, POS, largura=0.5, opacidade=0.3),
                   traco_nos(G, POS, cores_para_k(d), 8, hover_G, rotulos=False)], altura=620)

# rótulo só nos 7 do núcleo — 274 nomes seria ilegível
fig.add_trace(go.Scatter(
    x=[POS[v][0] for v in nucleo], y=[POS[v][1] for v in nucleo],
    mode="markers+text", text=[curto(v) for v in nucleo], textposition="top center",
    textfont=dict(size=10, color=LARANJA),
    marker=dict(color=LARANJA, size=11, line=dict(color="white", width=1.5)),
    hovertext=[f"<b>{curto(v)}</b><br>NÚCLEO · coreness {core[v]}<br>grau {grau[v]}" for v in nucleo],
    hoverinfo="text", showlegend=False))

botoes = []
for k in range(1, d + 1):
    sub = list(nx.k_core(G, k).nodes())
    hx = casco(sub, POS)
    formas = ([dict(type="path", path="M " + " L ".join(f"{x},{y}" for x, y in zip(*hx)) + " Z",
                    line=dict(color=AZUIS[6], dash="dot", width=1.5),
                    fillcolor="rgba(42,120,214,0.07)", layer="below")] if hx is not None else [])
    # o 3o elemento restringe o restyle ao traço 1 (os nós); sem ele o plotly
    # cicla as cores pelos traços e pinta as arestas por engano
    botoes.append(dict(label=f"{k}-core ({len(sub)} módulos)", method="update",
                       args=[{"marker.color": [cores_para_k(k)]},
                             {"title.text": f"{pacote} · em destaque: {k}-core "
                                            f"({len(sub)} de {G.number_of_nodes()})",
                              "shapes": formas},
                             [1]]))

fig.update_layout(title_text=f"{pacote} · em destaque: {d}-core ({len(nucleo)} de {G.number_of_nodes()})",
                  shapes=botoes[-1]["args"][1]["shapes"],
                  updatemenus=[dict(type="dropdown", x=0.01, y=0.99, xanchor="left", yanchor="top",
                                    active=len(botoes) - 1, buttons=botoes, bgcolor="white")])
fig.show()

**Três coisas que esta figura entrega, e que a tabela não entregava.**

**O núcleo não está no meio do desenho.** Essa é a observação mais útil aqui, porque desmente a intuição mais comum sobre k-core. O `spring_layout` posiciona por atração de arestas: um módulo com 227 vizinhos é puxado para o centro visual mesmo estando fora da cebola. **O olho não acha o núcleo, mas o algoritmo acha.**

**Os sete estão espalhados, mas próximos entre si.** O contorno tracejado do 5-core é pequeno perto do grafo todo. Eles se atraem porque estão de fato ligados uns aos outros.

**Desça o menu de 5 até 1 e observe o ritmo.** De 5 para 4 o destaque salta de 7 para 61 módulos — a mesma fronteira nítida que as barras mostraram, agora visível como área.

## 3 · Por que a resposta óbvia erra

Suponha que você não conhecesse k-core. A resposta natural para *"o que é perigoso mexer?"* é: **conte quantos módulos importam cada arquivo e pegue os primeiros**. É barato, é intuitivo, e é o que quase toda ferramenta de análise entrega.

Vamos medir o quanto essa resposta acerta.

In [11]:
top_grau = set(sorted(G.nodes(), key=lambda v: -grau[v])[:len(nucleo)])
acertos = top_grau & set(nucleo)

print(f"ranking por grau, {len(nucleo)} primeiros  ->  acertou {len(acertos)} de {len(nucleo)}\n")

comp = pd.DataFrame([{"módulo": curto(v), "grau": grau[v], "coreness": core[v],
                      "no núcleo?": "SIM" if core[v] == d else "não"}
                     for v in sorted(top_grau | set(nucleo), key=lambda v: -grau[v])])
print(comp.to_string(index=False))

ranking por grau, 7 primeiros  ->  acertou 1 de 7

                  módulo  grau  coreness no núcleo?
                networkx   227         5        SIM
                   utils   129         4        não
              algorithms    77         4        não
              generators    28         4        não
               exception    26         4        não
        utils.decorators    16         4        não
algorithms.approximation    14         4        não
           classes.graph     7         5        SIM
         classes.digraph     7         5        SIM
    classes.multidigraph     7         5        SIM
     classes.reportviews     6         5        SIM
      classes.multigraph     6         5        SIM
                 convert     5         5        SIM


In [ ]:
# popularidade contra emaranhamento
a = max((v for v in G if core[v] < d), key=lambda v: grau[v])          # o mais popular fora do núcleo
b = min(nucleo, key=lambda v: grau[v])                                  # o menos popular dentro

# de onde vem o grau de cada um: gente importando ele, ou ele importando gente?
_, mapa_nx = modulos_do_pacote(raiz)
importa = {nome: {x for x in imports_do_arquivo(cam, nome, pacote)
                  if x in G and x != nome} for cam, nome in mapa_nx.items()}
sai = lambda v: len(importa.get(v, ()))                       # quantos ELE importa
entra = lambda v: sum(1 for alvos in importa.values() if v in alvos)   # quantos importam ELE

print(f"{'':<12}{'grau':>6}{'core':>6}   {'importam ele':>13}{'ele importa':>13}   posição")
print("-" * 74)
for v, onde in ((a, "FORA do núcleo"), (b, "DENTRO do núcleo")):
    print(f"{curto(v):<12}{grau[v]:>6}{core[v]:>6}   {entra(v):>13}{sai(v):>13}   {onde}")

print(f"\n{curto(a)} é {grau[a] // grau[b]}x mais importado que {curto(b)} — e é o {curto(b)} que está preso.")

              grau  core    importam ele  ele importa   posição
--------------------------------------------------------------------------
utils          129     4             127            6   FORA do núcleo
convert          5     5               5            1   DENTRO do núcleo

utils é 25x mais importado que convert — e é o convert que está preso.


**A explicação é estrutural, não estatística.**

Olhe as duas colunas do meio da tabela acima, que separam o grau em suas duas origens. O `utils` é uma **folha**: das 129 ligações dele, **127 são outros módulos importando** e só **6** são ele importando. Dependência de mão única. Você pode reescrever o corpo das funções dele à vontade, enquanto as assinaturas não mudarem, ninguém quebra.

O `convert` está num **ciclo**: ele constrói objetos `Graph`, e o `Graph` chama ele. Grau baixo, dependência mútua. Mexeu num, tem que olhar o outro.

É exatamente o par `io_utils` × `core` dos cinco arquivos anteriores.

### Por que o k-core é a medida certa para esta pergunta

| a pergunta | a propriedade estrutural | a medida |
|---|---|---|
| "posso mexer aqui isoladamente?" | não, se eu estou preso num grupo mutuamente dependente | **k-core** |
| "quantos dependem de mim?" | popularidade, mão única | grau |

**Perigo, aqui, significa "não consigo alterar sem arrastar outros".** Isso é uma propriedade de **grupo**, não de vértice: depende de você estar dentro de um conjunto onde todo mundo depende de todo mundo. O k-core é a definição formal exata disso — *o maior subgrafo em que todo mundo tem pelo menos k vizinhos ali dentro*.

O grau é uma medida **local**: conta vizinhos, não conta reciprocidade nem grupo. Por isso ele não pode responder essa pergunta, por melhor que seja para outras.

## 4 · Bem arquitetada ou monólito?

Até aqui olhamos um pacote. O número fica muito mais útil quando você tem **com o que comparar**.

A medida que interessa é a **fração do pacote no núcleo**.

In [13]:
CANDIDATOS = ["networkx", "scipy", "numpy", "matplotlib", "pandas"]

def resumo(nome):
    r = raiz_do_pacote(nome)
    if r is None:
        return None
    try:
        _, g = grafo_de_dependencias(r)
    except Exception:
        return None
    if g.number_of_nodes() < 30:      # pacotes minúsculos não são comparáveis
        return None
    c = nx.core_number(g)
    dd = max(c.values())
    n = g.number_of_nodes()
    nuc = sum(1 for v in c.values() if v == dd)
    return {"pacote": nome, "módulos": n, "dependências": g.number_of_edges(),
            "degenerescência": dd, "núcleo": nuc, "% no núcleo": round(100 * nuc / n, 1),
            "_shells": pd.Series(c).value_counts().sort_index(), "_n": n, "_d": dd}

print("analisando pacotes instalados (pode levar um minuto)...")
res = [x for x in (resumo(p) for p in CANDIDATOS) if x]
tabela = (pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in res])
            .sort_values("% no núcleo").reset_index(drop=True))
tabela

analisando pacotes instalados (pode levar um minuto)...


,pacote,módulos,dependências,degenerescência,núcleo,% no núcleo
0,networkx,274,681,5,7,2.6
1,scipy,464,1103,6,13,2.8
2,numpy,229,644,7,12,5.2
3,matplotlib,145,786,12,21,14.5
4,pandas,298,2325,16,72,24.2


**A coluna que decide é a última.** E a distância entre os extremos não é sutil:

* **`networkx` e `scipy`: ~3% no núcleo.** Milhares de arquivos, e o miolo perigoso cabe numa mão.
* **`pandas`: 24% no núcleo, com degenerescência 16.** Um quarto do pacote (72 arquivos) está num grupo onde **cada um depende de pelo menos 16 outros do mesmo grupo**.

Vamos ver as duas formas lado a lado.

In [14]:
alvo_bom = next(r for r in res if r["pacote"] == "networkx")
alvo_mono = next(r for r in res if r["pacote"] == "pandas")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"{alvo_bom['pacote']} — {alvo_bom['% no núcleo']}% no núcleo (d={alvo_bom['_d']})",
    f"{alvo_mono['pacote']} — {alvo_mono['% no núcleo']}% no núcleo (d={alvo_mono['_d']})"))

for col, r in enumerate([alvo_bom, alvo_mono], start=1):
    s = r["_shells"]
    pct = 100 * s.values / r["_n"]
    cores = [LARANJA if k == r["_d"] else AZUIS[3] for k in s.index]
    fig.add_bar(x=s.index, y=pct, marker_color=cores, showlegend=False,
                hovertemplate="%{x}-shell<br>%{y:.1f}% dos módulos<extra></extra>", row=1, col=col)
    fig.update_xaxes(title_text="k (coreness)", dtick=1 if r["_d"] <= 8 else 2, row=1, col=col)

fig.update_yaxes(title_text="% dos módulos do pacote", row=1, col=1)
estilo(fig, "Onde está a massa do pacote? (laranja = a camada mais profunda)", 430).show()

for r in (alvo_bom, alvo_mono):
    faltando = [k for k in range(0, r["_d"] + 1) if k not in r["_shells"].index]
    print(f"{r['pacote']:<11} camadas vazias: {faltando if faltando else 'nenhuma'}")

networkx    camadas vazias: nenhuma
pandas      camadas vazias: [13]


**A forma conta a história toda.**

No **`networkx`**, a massa está no meio e a barra laranja — a camada mais profunda — é um toco de 2,6%. A leitura de manutenção: *quase todo arquivo deste pacote é seguro de mexer isoladamente.*

No **`pandas`**, a barra laranja é **a maior de todas**. A distribuição sobe no fim em vez de cair. A leitura: *um quarto do pacote é uma peça única, e não existe "mexer só nesta parte".*

Repare também na camada vazia do `pandas` (a shell 13 não existe). Isso significa que ao descascar a camada 12, o que sobrou já era tão coeso que todo mundo tinha 14+ vizinhos internos. Não há degradação gradual, há um platô e depois um nucleo.

### Para que isso serve, na prática

Essa comparação pode trazer três usos diretos:

**1. Decidir se você vai conseguir contribuir.** Antes de aceitar uma tarefa "adicione um recurso X nesta biblioteca", rode a decomposição. Núcleo abaixo de ~5% significa que existe uma borda onde dá para trabalhar sem entender o sistema inteiro. Acima de ~20%, provavelmente não existe, você vai ter que entender o miolo primeiro, e isso muda a estimativa de prazo.

**2. Escolher entre dependências equivalentes.** Duas bibliotecas resolvem seu problema. Se você espera precisar estendê-la, corrigir bugs ou fazer fork, a fração no núcleo é um indicador barato de quanto isso vai doer.

**3. Acompanhar a própria dívida ao longo do tempo.** A degenerescência é determinística e sem parâmetro, dá para rodar a cada commit e plotar. Se ela sobe mês a mês, sua arquitetura está apodrecendo e você tem número, não opinião, para levar à reunião.

**Uma ressalva importante:** este critério só compara pacotes de porte parecido. Um pacote de 15 módulos vai dar 60% no núcleo quase sempre, porque com poucos arquivos quase todo grafo conexo vira uma camada só. Por isso a função acima descarta pacotes com menos de 30 módulos.

## 5 · O k-core não é a única coisa que você deve olhar

Seria desonesto parar aqui deixando a impressão de que k-core substitui as outras métricas. Ele responde **uma** pergunta muito bem. Existe outro risco de manutenção que ele **não** enxerga.

In [15]:
CC = G.subgraph(max(nx.connected_components(G), key=len)).copy()  # métricas de caminho pedem conexo
inter = nx.betweenness_centrality(CC)
art = set(nx.articulation_points(CC))

print(f"pontos de articulação (remova um e o grafo se PARTE): {len(art)}")
print(f"desses, quantos estão no núcleo: {sum(1 for v in art if core[v] == d)}\n")

print("os 10, do menos ao mais profundo:")
print(f"{'módulo':<44}{'coreness':>9}{'grau':>6}")
print("-" * 60)
for v in sorted(art, key=lambda v: (core[v], grau[v])):
    marca = "   <-- no núcleo" if core[v] == d else ""
    print(f"{curto(v):<44}{core[v]:>9}{grau[v]:>6}{marca}")

r = pd.Series(core).corr(pd.Series(inter), method="spearman")
print(f"\ncorrelação (Spearman) entre coreness e intermediação: {r:.2f}")

pontos de articulação (remova um e o grafo se PARTE): 10
desses, quantos estão no núcleo: 1

os 10, do menos ao mais profundo:
módulo                                       coreness  grau
------------------------------------------------------------
algorithms.traversal.breadth_first_search           1     2
algorithms.centrality.betweenness                   3     7
algorithms.isomorphism                              3     8
linalg                                              3     8
algorithms.community.modularity_max                 4     5
algorithms.approximation                            4    14
utils.decorators                                    4    16
algorithms                                          4    77
utils                                               4   129
networkx                                            5   227   <-- no núcleo



correlação (Spearman) entre coreness e intermediação: 0.50


**São dois riscos de engenharia diferentes, e eles quase não se sobrepõem.**

Um **ponto de articulação** é um módulo que, se removido, parte o grafo em pedaços desconexos. Este pacote tem 10, e **apenas um** deles está no núcleo — os outros nove a decomposição classificaria como território tranquilo.

| | acha | o risco que isso representa | custo |
|---|---|---|---|
| **k-core** | o que está **emaranhado** | não consigo alterar isto isoladamente | 0,3 ms |
| **intermediação / articulação** | o que é **ponte** | se isto sair, um pedaço inteiro para | ~150 ms |

**Como isso contribui para a solução do problema.** A pergunta "no que devo ter cuidado ao manter esta biblioteca" tem **duas** respostas, e você precisa das duas:

* **Do k-core** vem a lista do que exige *contexto*: não dá para abrir só este arquivo, você tem que carregar os outros seis na cabeça. É risco de **esforço e regressão**.
* **Dos pontos de articulação** vem a lista do que exige *cuidado com terceiros*: mexer aqui pode desligar um subsistema inteiro que nem parecia relacionado. É risco de **alcance**.

Um módulo pode estar numa lista, na outra, nas duas, ou em nenhuma. A correlação entre as duas medidas é **moderada — 0,50**: elas concordam em parte, mas discordam com frequência mais que suficiente para que uma não substitua a outra. Repare no extremo da tabela acima: o `breadth_first_search` tem **coreness 1 e grau 2** — pela decomposição, é periferia pura — e mesmo assim, se ele sair, o grafo se parte. A decisão prática é simples: **rode o k-core sempre** (custa nada) e **rode a intermediação quando a pergunta for sobre ponte** (custa 500× mais).

### A partição inteira, desenhada

Uma última figura, porque ela deixa explícito o que a tabela de barras já dizia: as k-shells **particionam** o pacote. Aqui a posição de cada módulo não vem das arestas cada um é colocado no **anel da sua camada**, do núcleo para fora.

In [16]:
shells = [s for s in ([v for v in G if core[v] == k] for k in range(d, -1, -1)) if s]
POS_ANEL = nx.shell_layout(G, nlist=shells)

cor_shell = {k: AZUIS[max(0, min(len(AZUIS) - 1, 1 + int(round(k / max(d, 1) * (len(AZUIS) - 2)))))]
             for k in range(0, d + 1)}

fig = figura_rede([traco_arestas(G, POS_ANEL, largura=0.35, opacidade=0.18),
                   traco_nos(G, POS_ANEL, [cor_shell[core[n]] for n in nos], 8, hover_G, rotulos=False)],
                  f"{pacote} · um anel por k-shell", altura=640)

for s in shells:
    k = core[s[0]]
    rr = float(np.mean([math.hypot(*POS_ANEL[v]) for v in s]))
    fig.add_shape(type="circle", x0=-rr, y0=-rr, x1=rr, y1=rr, layer="below",
                  line=dict(color=CINZA_ARESTA, width=1, dash="dot"))
    fig.add_annotation(x=0, y=rr, text=f"<b>{k}-shell</b> · {len(s)}", showarrow=False, yshift=11,
                       font=dict(size=11, color=CINZA_TEXTO), bgcolor=SUPERFICIE, opacity=0.9)
fig.show()

**O que esta figura acrescenta.** Compare com a da Seção 2.2: são o mesmo grafo e a mesma informação, com uma troca de papéis. Lá a posição vinha das arestas e a cor carregava a coreness; aqui **a posição é a coreness** e as arestas viraram fundo.

Por isso a cebola aparece aqui e não aparecia lá. Ela não estava escondida aquele layout simplesmente não desenhava essa pergunta. **Escolher o layout é escolher qual pergunta a figura responde.**

E repare que as arestas cruzam os anéis o tempo todo, sem respeitar a hierarquia. Isso não é ruído: mostra que módulos da periferia se ligam **direto** ao núcleo, sem passar pelas camadas do meio. A cebola é uma hierarquia de **coesão**, não de distância — um arquivo periférico pode muito bem importar `Graph` diretamente.

## 6 · Ir além: do número à recomendação

Uma métrica só vira ferramenta quando vira **decisão**. Até aqui temos uma coluna de números; o que um time precisa é de uma frase por arquivo dizendo o que fazer.

A tradução abaixo cruza as duas medidas da Seção 5 e devolve um **playbook de manutenção**:

| situação | papel | recomendação |
|---|---|---|
| coreness = degenerescência | núcleo | teste de integração; revisão cuidadosa; não extrair sozinho |
| coreness ≥ metade da degenerescência | meio | atenção em mudanças de interface |
| coreness ≤ 1 | folha | seguro; extraível quase de graça |
| é ponto de articulação | **+ ponte** | ponto único de falha, independentemente da coreness |

In [17]:
def playbook(G, core, articulacoes=()):
    d = max(core.values())
    art = set(articulacoes)
    linhas = []
    for v in G.nodes():
        c = core[v]
        if c == d:
            papel, acao = "núcleo", "teste de integração; não extrair sozinho"
        elif c >= d / 2:
            papel, acao = "meio", "atenção em mudanças de interface"
        elif c <= 1:
            papel, acao = "folha", "seguro; extraível quase de graça"
        else:
            papel, acao = "casca", "baixo risco"
        if v in art:
            papel, acao = papel + " + ponte", acao + "; PONTO ÚNICO DE FALHA"
        linhas.append({"módulo": curto(v), "coreness": c, "grau": G.degree(v),
                       "papel": papel, "recomendação": acao})
    return (pd.DataFrame(linhas)
              .sort_values(["coreness", "grau"], ascending=False)
              .reset_index(drop=True))

rel = playbook(G, core, art)

print("quantos módulos em cada papel:")
print(rel.papel.value_counts().to_string())
print(f"\ntotal que exige cuidado (núcleo ou ponte): "
      f"{(rel.papel.str.contains('núcleo|ponte')).sum()} de {len(rel)}")
rel[rel.papel.str.contains("núcleo|ponte")].head(14)

quantos módulos em cada papel:
papel
meio              147
casca              85
folha              26
meio + ponte        8
núcleo              6
núcleo + ponte      1
folha + ponte       1

total que exige cuidado (núcleo ou ponte): 16 de 274


,módulo,coreness,grau,papel,recomendação
0,networkx,5,227,núcleo + ponte,teste de integração; não extrair sozinho; PONT...
1,classes.multidigraph,5,7,núcleo,teste de integração; não extrair sozinho
2,classes.graph,5,7,núcleo,teste de integração; não extrair sozinho
3,classes.digraph,5,7,núcleo,teste de integração; não extrair sozinho
4,classes.multigraph,5,6,núcleo,teste de integração; não extrair sozinho
5,classes.reportviews,5,6,núcleo,teste de integração; não extrair sozinho
6,convert,5,5,núcleo,teste de integração; não extrair sozinho
7,utils,4,129,meio + ponte,atenção em mudanças de interface; PONTO ÚNICO ...
8,algorithms,4,77,meio + ponte,atenção em mudanças de interface; PONTO ÚNICO ...
11,utils.decorators,4,16,meio + ponte,atenção em mudanças de interface; PONTO ÚNICO ...


A tabela responde a pergunta que abriu o notebook em forma acionável — para cada um dos 274 arquivos, o que fazer se você precisar mexer nele. E os números fecham o argumento: **de 274 módulos, só uma dúzia e meia exige cuidado real**. O resto você abre e altera sem carregar o sistema inteiro na cabeça.

**Três formas de levar isso para o dia a dia:**

**No CI.** A decomposição custa O(n + m). Um passo que roda a cada commit e falha se a degenerescência subir custa menos que um teste unitário — e transforma "acho que a arquitetura está piorando" em um alerta objetivo.

**Na revisão de código.** Um PR que toca um arquivo do núcleo merece revisor sênior e teste de integração. Um PR que toca só folhas não. Isso é uma regra que dá para automatizar a partir desta tabela.

**Na entrada de gente nova.** A pergunta "por onde eu começo a ler?" tem resposta: pelos módulos do núcleo, que são o esqueleto conceitual do pacote. Ler `Graph`, `convert` e `reportviews` ensina mais sobre o NetworkX do que ler cinquenta algoritmos da periferia.

### Para levar adiante

* **Rode no seu próprio repositório.** Troque `ALVO` na Seção 2 pelo caminho do seu projeto (`grafo_de_dependencias("/caminho/do/projeto")`) e olhe o que sai. O resultado costuma surpreender quem escreveu o código.
* **Acompanhe ao longo do tempo.** Rode em commits antigos e veja se a degenerescência subiu.
* **Ajuste as regras do playbook** ao seu contexto — os cortes acima são um ponto de partida, não uma lei.

## Resumo

| pergunta | resposta |
|---|---|
| O que é perigoso mexer nesta biblioteca? | os módulos do **k-core** máximo — 7 de 274 no NetworkX |
| Por que não basta contar quem me importa? | grau mede popularidade (mão única); perigo aqui é **dependência mútua** |
| Quanto custa descobrir? | **O(n + m)** — roda em milissegundos, cabe no CI |
| Como comparo duas bibliotecas? | pela **fração do pacote no núcleo** — 2,6% (networkx) contra 24,2% (pandas) |
| O k-core basta? | não: ele acha o **emaranhado**; pontos de articulação acham a **ponte** |
| Qual é o produto final? | o **playbook** — uma recomendação por arquivo |